# Session 3.1 -- Naive RAG

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Explain** the Retrieve-Augment-Generate pattern and articulate why retrieval quality bounds answer quality
2. **Trace** the naive RAG pipeline end-to-end -- from querying ChromaDB, through constructing an augmented prompt, to generating an answer with source citations via Claude -- and inspect results at each stage
3. **Identify** semantic similarity pitfalls -- semantic drift, keyword-heavy queries, ambiguous intent, and the "lost in the middle" problem -- and predict when naive RAG will fail

## What You Are Working With

- `src/s4_retrieval/naive.py` -- Naive RAG module with `naive_retrieve()`, `build_prompt()`, `naive_rag()` (provided complete)
- `src/s0_generation/generate.py` -- Claude API functions from Session 1.1 (provided complete)
- `src/s2_embeddings/embed.py` -- Voyage AI embedding functions from Session 2.1 (provided complete)
- `src/s3_ingestion/store.py` -- ChromaDB storage from Session 2.2 (provided complete)
- `data/northbrook/` -- Corpus of memos, meeting notes, financial reports, and policies from Northbrook Partners

You **import and use** the pre-built modules. You do not need to build them from scratch.

**Prerequisite:** Your ChromaDB collection must be populated from Session 2.2. If it is empty, run the ingestion notebook first.

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")

from dotenv import load_dotenv
load_dotenv()

In [ ]:
from src.s4_retrieval.naive import naive_rag, naive_retrieve, build_prompt, RAGResponse
from src.s3_ingestion.store import get_collection

print("All imports loaded successfully.")

In [ ]:
# Verify ChromaDB is populated from Session 2.2
collection = get_collection()
count = collection.count()
print(f"Collection '{collection.name}' contains {count} chunks.")

if count == 0:
    print("\nWARNING: Your collection is empty!")
    print("Run the Session 2.2 notebook first to populate ChromaDB.")
else:
    print(f"\nReady to go. {count} chunks available for retrieval.")

---

## 2. RAG Overview -- Retrieve, Augment, Generate

RAG stands for **Retrieval Augmented Generation**. The idea is simple: before asking a model to answer a question, **look up relevant information first** and include it in the prompt.

- **Not fine-tuning** -- we do not change the model
- **Not prompt engineering alone** -- we dynamically select what goes into the prompt
- RAG = "look it up, then answer" -- the model becomes a reasoning engine over YOUR data

### The Pipeline

```
                  +-----------+
  Question -----> | RETRIEVE  |  Embed the question, query ChromaDB
                  +-----------+  for top-k most similar chunks
                       |
                       v
                  +-----------+
    Chunks -----> | AUGMENT   |  Inject retrieved chunks into a
                  +-----------+  structured prompt with grounding
                       |         instructions
                       v
                  +-----------+
   Prompt  -----> | GENERATE  |  Send the augmented prompt to Claude
                  +-----------+  and return the answer + sources
                       |
                       v
               Answer + Sources
```

### Three Truths About RAG

These came from our Session 2.2 closing questions:

1. **Similar != Useful** -- Semantic similarity measures direction in embedding space, not usefulness for a specific question
2. **Context present != Answer grounded** -- Claude will try to answer even if the context is insufficient, filling gaps from training data
3. **Trust requires evidence** -- You must see the retrieved chunks, their scores, and compare the answer against the source text

### Why "Naive"?

Naive RAG = pure semantic similarity retrieval, no filtering, no re-ranking, no query transformation. Every query gets the same retrieval logic regardless of question type. This is the baseline. Session 3.2 raises the ceiling with metadata filtering.

---

## 3. First RAG Query

Let's run the full pipeline end-to-end. One function call: question in, grounded answer out.

In [ ]:
# Your first RAG call
response = naive_rag("What is Northbrook Partners' vacation policy?")

print("=" * 70)
print("QUESTION")
print("=" * 70)
print(response.question)
print()
print("=" * 70)
print("ANSWER")
print("=" * 70)
print(response.answer)

In [ ]:
# Inspect the RAGResponse object -- everything is visible
print(f"Model:         {response.model}")
print(f"Input tokens:  {response.input_tokens}")
print(f"Output tokens: {response.output_tokens}")
print(f"Sources:       {len(response.sources)} chunks retrieved")
print()

# List the sources with their similarity scores
print("--- Retrieved Sources ---")
for i, src in enumerate(response.sources):
    source_name = src["metadata"].get("source", "unknown")
    doc_type = src["metadata"].get("doc_type", "unknown")
    score = src["score"]
    print(f"  [{i+1}] {score:.4f}  {source_name}  (type: {doc_type})")

print()
print("Higher scores mean the chunk is more similar to the question.")
print("But similar does not guarantee useful -- we will test that.")

In [ ]:
# Look at the actual text of the top source
top_source = response.sources[0]

print(f"--- Top Source (score: {top_source['score']:.4f}) ---")
print(f"From: {top_source['metadata'].get('source', 'unknown')}")
print()
print(top_source["text"])

**What just happened:**

1. The question was embedded using Voyage AI (same model that embedded the documents)
2. ChromaDB returned the 5 most similar chunks
3. Those chunks were formatted into a prompt with grounding instructions
4. Claude read the chunks and produced an answer citing the sources
5. The response includes the answer, sources, scores, and token counts

That is RAG. Now let's look under the hood.

---

## 4. Under the Hood -- Step Through the Pipeline

The `naive_rag()` function wraps three stages. Let's call each one separately to see what happens at every step.

### Stage 1: Retrieve

`naive_retrieve()` embeds the question and queries ChromaDB for the top-k most similar chunks.

In [ ]:
# Step 1: Retrieve chunks
question = "What is Northbrook Partners' vacation policy?"
sources = naive_retrieve(question, top_k=5)

print(f"Retrieved {len(sources)} chunks for: '{question}'\n")

for i, src in enumerate(sources):
    score = src["score"]
    source_name = src["metadata"].get("source", "unknown")
    doc_type = src["metadata"].get("doc_type", "unknown")
    chunk_idx = src["metadata"].get("chunk_index", "?")
    text_preview = src["text"][:150].replace("\n", " ")

    print(f"--- Chunk {i+1} [similarity: {score:.4f}] ---")
    print(f"Source: {source_name}  |  Type: {doc_type}  |  Chunk: {chunk_idx}")
    print(f"Text:   {text_preview}...")
    print()

**What to notice:**

- The chunks are sorted by similarity score (highest first)
- ChromaDB returns cosine distances. The module converts them: **similarity = 1 - distance**
- Some chunks are highly relevant. Others may be topically related but not directly useful
- The metadata tells you exactly which document each chunk came from

**Key question:** Are ALL of these chunks actually useful for answering the question? Or did some drift in because they share vocabulary?

### Stage 2: Augment

`build_prompt()` takes the question and retrieved sources, and constructs a grounded prompt for Claude.

In [ ]:
# Step 2: Build the augmented prompt
system_prompt, user_message = build_prompt(question, sources)

print("=" * 70)
print("SYSTEM PROMPT")
print("=" * 70)
print(system_prompt)
print()
print("=" * 70)
print("USER MESSAGE (first 1000 characters)")
print("=" * 70)
print(user_message[:1000])
if len(user_message) > 1000:
    print(f"\n... [{len(user_message) - 1000} more characters]")

In [ ]:
# Let's see the FULL user message to understand what Claude receives
print("FULL USER MESSAGE:")
print("=" * 70)
print(user_message)

**Read through the prompt carefully.** This is what Claude actually sees. Notice:

- The **system prompt** tells Claude to answer using ONLY the provided context and to say "I don't have enough information" when context is insufficient. This is the grounding instruction.
- Each chunk includes its **source name** and **similarity score** in the header -- this lets Claude cite sources.
- The chunks are separated by `---` dividers for clarity.
- The question appears **after** the context. Claude reads the evidence before answering.

Without the grounding instruction, Claude will happily fill in gaps from its training data. With it, Claude is more likely to say "I don't know" when the context is insufficient.

### Stage 3: Generate

The augmented prompt goes to Claude, and Claude produces an answer grounded in the retrieved context. This is handled inside `naive_rag()` by calling `call_claude_with_usage()`. We already saw the result in Section 3 above.

**Generation is the easy part. The hard part is everything before it.**

In [ ]:
# Try a different question -- step through all three stages
question_2 = "What is the dress code at Northbrook Partners?"

# Retrieve
sources_2 = naive_retrieve(question_2, top_k=5)
print(f"Question: {question_2}")
print(f"\nRetrieved {len(sources_2)} chunks:")
for i, s in enumerate(sources_2):
    print(f"  [{i+1}] {s['score']:.4f}  {s['metadata'].get('source', '?')}")

# Augment
sys_2, msg_2 = build_prompt(question_2, sources_2)
print(f"\nPrompt length: {len(sys_2) + len(msg_2):,} characters")

# Generate (via full pipeline)
response_2 = naive_rag(question_2)
print(f"\nAnswer:\n{response_2.answer}")
print(f"\nTokens: {response_2.input_tokens} in, {response_2.output_tokens} out")

---

## 5. Answer vs Evidence Panel

How do you verify that an AI-generated answer actually came from your sources? You compare them side by side.

The helper function below takes a `RAGResponse` and prints Claude's answer alongside the actual chunk text. This makes unsupported claims immediately visible.

In [ ]:
import textwrap


def compare_answer_to_evidence(rag_response: RAGResponse) -> None:
    """Display Claude's answer alongside the actual retrieved evidence.

    This helper makes it easy to spot unsupported claims. If a claim
    appears in the answer but NOT in any chunk, Claude made it up.

    Args:
        rag_response: A RAGResponse from naive_rag() or similar.
    """
    divider = "=" * 70

    print(divider)
    print("QUESTION")
    print(divider)
    print(rag_response.question)
    print()

    print(divider)
    print("CLAUDE'S ANSWER")
    print(divider)
    # Break the answer into individual sentences for easier comparison
    answer_text = rag_response.answer
    print(answer_text)
    print()

    print(divider)
    print("RETRIEVED EVIDENCE (what Claude was given)")
    print(divider)
    for i, src in enumerate(rag_response.sources):
        source_name = src["metadata"].get("source", "unknown")
        score = src["score"]
        print(f"\n--- Source {i+1}: {source_name} [score: {score:.4f}] ---")
        # Wrap text for readability
        wrapped = textwrap.fill(src["text"], width=80)
        print(wrapped)

    print()
    print(divider)
    print("VERIFICATION CHECKLIST")
    print(divider)
    print("For each claim in the answer above, check:")
    print("  [?] Does this claim appear in one of the source chunks?")
    print("  [?] Is the claim accurately represented (not taken out of context)?")
    print("  [?] Are there claims in the answer with NO supporting evidence above?")
    print()
    print("If a claim appears in the answer but NOT in any chunk -- Claude made it up.")
    print("If a claim merges information from chunks that should not be combined --")
    print("that is also a hallucination.")

In [ ]:
# Run it on our first response -- the vacation policy question
compare_answer_to_evidence(response)

**Walk through the comparison above.**

- Read each claim in Claude's answer
- Find the supporting text in the retrieved sources
- Are there any claims that do NOT appear in any source?

For a straightforward factual question like the vacation policy, the answer should be well-grounded. But keep this helper ready -- we are about to see cases where the comparison reveals serious problems.

---

## 6. The Confidence Mirage

This is the most important demo of the session.

Low similarity scores are easy to catch -- you see the problem immediately. The dangerous failures are the ones where:

- Similarity scores are **high** (0.80+)
- Claude produces a **confident, well-cited** answer
- The answer is **wrong**

### The Setup

The Northbrook corpus contains three documents relevant to PTO:

1. **vacation_policy_2025.md** -- The new 2025 policy that says the carryover limit is **10 days** (increased from 4)
2. **memo_policy_correction.md** -- A correction memo from February 2025 stating that the **old 4-day limit still applies for 2025**, and the 10-day cap does not take effect until January 1, 2026
3. **vacation_policy_2023.md** -- The old policy with a **4-day carryover limit** and 17 vacation days

What happens when we ask about the current carryover limit?

In [ ]:
# THE CONFIDENCE MIRAGE
#
# This query is designed to expose the most dangerous RAG failure:
# high scores + confident answer + wrong

mirage_query = "What is the PTO carryover limit for 2025?"
mirage_response = naive_rag(mirage_query)

print("=" * 70)
print("THE CONFIDENCE MIRAGE")
print("=" * 70)
print(f"\nQuestion: {mirage_query}")
print(f"\nAnswer:")
print(mirage_response.answer)
print()
print("--- Sources ---")
for i, src in enumerate(mirage_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

**Read the answer above carefully.**

- Are the similarity scores high? (They should be -- all these documents are about PTO.)
- Does Claude sound confident?
- Does Claude cite sources?

Now let's look at what the chunks actually say.

In [ ]:
# Now compare the answer to the actual evidence
compare_answer_to_evidence(mirage_response)

### What Went Wrong?

The corpus contains **conflicting information** about the carryover limit:

| Document | Says |
|----------|------|
| `vacation_policy_2025.md` | Carryover limit is **10 days** (increased from 4) |
| `memo_policy_correction.md` | The 10-day cap does **NOT** take effect until Jan 1, 2026. For 2025, the old **4-day limit** still applies |
| `vacation_policy_2023.md` | Carryover limit is **4 days** |

**The correct answer for 2025 is 4 days** (per the correction memo). But naive RAG retrieves chunks from all three documents because they are all semantically similar to the question. Claude then has to reconcile conflicting context.

The typical failure mode: Claude either picks the 10-day number (from the 2025 policy) because it appears to be the newer document, or merges information from multiple sources without flagging the contradiction.

**This is the Confidence Mirage:**
- High similarity scores (the retrieval "worked")
- Confident, well-cited answer (Claude "did its job")
- **Wrong answer** (the system failed silently)

The scores did not warn you. The citations did not warn you. The only way to catch this is to **read the chunks and compare them to the answer**.

In [ ]:
# Let's look at the retrieval more closely
# Which documents did naive_retrieve pull back, and what do they say?

mirage_sources = naive_retrieve(mirage_query, top_k=5)

print(f"Retrieval results for: '{mirage_query}'\n")

for i, src in enumerate(mirage_sources):
    source_name = src["metadata"].get("source", "unknown")
    score = src["score"]
    print(f"=== Chunk {i+1}: {source_name} [score: {score:.4f}] ===")
    print(src["text"])
    print()

**Key takeaway:** Naive RAG treats all documents equally. It has no way to know that the correction memo overrides the policy document. It has no way to reason about recency, authority, or document relationships. It just finds the most similar chunks and hands them to Claude.

In production, this is how you get confident, well-sourced, **wrong** answers. The dangerous failures are the ones that look correct.

---

## 7. Failure Gallery

Now it is your turn. Below are five failure categories. Your goal: **find at least 3 failures from at least 3 different categories.**

For each failure, document:
1. The **question** you asked
2. The **top-3 retrieved chunks** (source and score)
3. **Claude's answer**
4. Your **analysis** of what went wrong

Use `compare_answer_to_evidence()` to inspect each one.

**These failures become your Lab 2 test set.** Take them seriously.

### Failure Categories

| Category | Description | What to Try |
|----------|-------------|-------------|
| **Semantic Drift** | Chunks are topically related but do not answer the question | Questions about specific details when corpus has general overviews; different vocabulary ("PTO" vs "vacation") |
| **Keyword-Heavy Queries** | Terms in the question skew results toward wrong chunks | Questions mentioning proper nouns that appear in irrelevant documents; ambiguous terms |
| **Hallucination Despite Context** | Claude produces an answer not supported by the chunks | Questions where chunks have partial info; chunks from different time periods that Claude merges |
| **Top-K Sensitivity** | Answer quality changes significantly with different k values | Same question with top_k=1 vs 3 vs 10; questions where more chunks introduce noise |
| **Unanswerable Questions** | The corpus genuinely cannot answer this | Questions about topics not in the corpus; specific dates/people/numbers not in any document |

### Failure 1: Wrong Version

When multiple versions of a document exist, naive RAG may pull the wrong one -- or merge them.

In [ ]:
# Failure 1: Wrong version
# The corpus has vacation_policy_2023.md AND vacation_policy_2025.md
# Which one does naive RAG retrieve?

wrong_version_query = "How many vacation days do employees get?"
wrong_version_response = naive_rag(wrong_version_query)

print(f"Question: {wrong_version_query}\n")
print(f"Answer:\n{wrong_version_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(wrong_version_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

In [ ]:
# Side-by-side comparison
compare_answer_to_evidence(wrong_version_response)

**Your analysis:** (Write your observations here)

- Which policy versions appeared in the results?
- Did Claude merge information from the 2023 and 2025 policies?
- What is the correct answer? (2025 policy: 20 days; 2023 policy: 17 days, or 20 with 5+ years tenure)
- What would you need to fix this? (Hint: what if you could filter by year?)

### Failure 2: Temporal Confusion

Three documents contain conflicting information about the same topic from different time periods.

In [ ]:
# Failure 2: Temporal confusion -- 3-way conflict
# vacation_policy_2023.md: 4-day carryover, 17 days vacation
# vacation_policy_2025.md: 10-day carryover, 20 days vacation
# memo_policy_correction.md: actually still 4-day carryover for 2025

temporal_query = "What is the maximum number of vacation days I can carry over to next year?"
temporal_response = naive_rag(temporal_query)

print(f"Question: {temporal_query}\n")
print(f"Answer:\n{temporal_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(temporal_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

In [ ]:
compare_answer_to_evidence(temporal_response)

**Your analysis:** (Write your observations here)

- Did Claude acknowledge the contradiction between documents?
- What number did Claude give? Was it correct?
- The **correct** answer depends on timing: for 2025 the carryover limit is still 4 days; starting 2026, it becomes 10 days
- Naive RAG has no concept of "which document is authoritative" or "which document is more recent"

### Failure 3: Missing Context (Answer Spans Multiple Chunks)

What happens when the answer requires information from two chunks that are not both in the top-k?

In [ ]:
# Failure 3: Missing context
# The professional development budget ($3,200) is in the employee handbook.
# But the expense reimbursement process is in the expense policy.
# A question that spans both may only retrieve one.

missing_context_query = "How do I get reimbursed for a professional development conference?"
missing_context_response = naive_rag(missing_context_query)

print(f"Question: {missing_context_query}\n")
print(f"Answer:\n{missing_context_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(missing_context_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

In [ ]:
compare_answer_to_evidence(missing_context_response)

**Your analysis:** (Write your observations here)

- Did the retrieval pull chunks from both the employee handbook AND the expense policy?
- Does the answer combine the budget amount ($3,200) with the reimbursement process?
- If one document was missing from the results, did Claude fill in the gap or acknowledge it?
- This is the "information scattered across documents" problem -- naive RAG retrieves by similarity, not by completeness

### Failure 4: Over-Retrieval (Vague Query)

What happens when the query is too broad and retrieves chunks from many unrelated documents?

In [ ]:
# Failure 4: Over-retrieval with a vague query
# A broad question that could match many documents

vague_query = "What are the important company policies?"
vague_response = naive_rag(vague_query)

print(f"Question: {vague_query}\n")
print(f"Answer:\n{vague_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(vague_response.sources):
    source_name = src["metadata"].get("source", "unknown")
    doc_type = src["metadata"].get("doc_type", "unknown")
    print(f"  [{i+1}] {src['score']:.4f}  {source_name}  (type: {doc_type})")

In [ ]:
compare_answer_to_evidence(vague_response)

**Your analysis:** (Write your observations here)

- How many different documents appeared in the sources?
- Are the similarity scores all relatively close (indicating no chunk is clearly "the right one")?
- Is the answer a shallow summary that does not deeply address any one topic?
- What would make this question answerable? (More specific query, narrower scope)

### Failure 5: Unanswerable Question

The corpus does not contain the information needed to answer this question. Does Claude admit it?

In [ ]:
# Failure 5: Unanswerable question -- topic not in the corpus
unanswerable_query = "What was the company's Q3 2024 revenue?"
unanswerable_response = naive_rag(unanswerable_query)

print(f"Question: {unanswerable_query}\n")
print(f"Answer:\n{unanswerable_response.answer}\n")
print("--- Sources ---")
for i, src in enumerate(unanswerable_response.sources):
    print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")

print("\nNotice the similarity scores. Are they lower than for answerable questions?")
print("Did Claude say 'I don't have enough information' or did it fabricate an answer?")

In [ ]:
compare_answer_to_evidence(unanswerable_response)

**Your analysis:** (Write your observations here)

- Did Claude say "I don't have enough information" or did it fabricate a number?
- What were the similarity scores? Were they noticeably lower?
- Did the grounding instruction in the system prompt do its job?
- If Claude DID give a number, where did it come from? (Its training data, not your corpus.)

### Your Own Failure Experiments

Use the cells below to run your own failure experiments. Try to find failures from categories you have not covered yet.

In [ ]:
# Experiment: Your own failure query
# Try different failure categories: semantic drift, keyword-heavy,
# hallucination despite context, top-k sensitivity, unanswerable

my_query = ""  # <-- Replace with your query

if my_query:
    my_response = naive_rag(my_query)
    compare_answer_to_evidence(my_response)
else:
    print("Replace the empty string above with your query and run this cell.")

In [ ]:
# Experiment: Top-K sensitivity
# Run the same question with different top_k values and compare

topk_query = "What is the Navigator onboarding program?"  # <-- Change if you like

for k in [1, 3, 5, 10]:
    print(f"{'=' * 70}")
    print(f"top_k = {k}")
    print(f"{'=' * 70}")

    topk_response = naive_rag(topk_query, top_k=k)
    print(f"\nAnswer:\n{topk_response.answer}")
    print(f"\nSources:")
    for i, src in enumerate(topk_response.sources):
        print(f"  [{i+1}] {src['score']:.4f}  {src['metadata'].get('source', 'unknown')}")
    print(f"\nTokens: {topk_response.input_tokens} in, {topk_response.output_tokens} out")
    print()

**Top-K observations:** (Write your observations here)

- Did the answer change with different k values?
- At what point did additional chunks introduce noise instead of adding value?
- Look at the token counts -- how does cost scale with k?
- What is the sweet spot for this question?

In [ ]:
# Experiment: Another failure query (use this cell for additional experiments)

my_query_2 = ""  # <-- Replace with your query

if my_query_2:
    my_response_2 = naive_rag(my_query_2)
    compare_answer_to_evidence(my_response_2)
else:
    print("Replace the empty string above with your query and run this cell.")

---

## 8. Scoring Retrieval Quality

So far we have been eyeballing results. Can we measure retrieval quality more systematically?

A simple metric: **Did the right document appear in the top-k results? What rank was it?**

This is called **Hit Rate** (did the correct source appear?) and **Mean Reciprocal Rank** (how high did it rank?).

In [ ]:
# Define a small test set with known-good source documents
# For each question, we know which document SHOULD be in the top-k

test_queries = [
    {
        "question": "What is the vacation policy?",
        "expected_source": "vacation_policy_2025.md",
    },
    {
        "question": "What is the dress code?",
        "expected_source": "employee_handbook.md",
    },
    {
        "question": "What is the meal reimbursement limit?",
        "expected_source": "expense_policy.md",
    },
    {
        "question": "What is the remote work policy?",
        "expected_source": "remote_work_policy.md",
    },
    {
        "question": "What was corrected about the PTO carryover?",
        "expected_source": "memo_policy_correction.md",
    },
]

print(f"Running {len(test_queries)} test queries...\n")
print(f"{'Question':<55} {'Expected Source':<30} {'Rank':>5} {'Hit?':>5}")
print("-" * 100)

hits = 0
reciprocal_ranks = []

for test in test_queries:
    sources = naive_retrieve(test["question"], top_k=5)

    # Find the rank of the expected source
    rank = None
    for i, src in enumerate(sources):
        if src["metadata"].get("source") == test["expected_source"]:
            rank = i + 1
            break

    hit = rank is not None
    if hit:
        hits += 1
        reciprocal_ranks.append(1 / rank)
    else:
        reciprocal_ranks.append(0)

    rank_str = str(rank) if rank else "MISS"
    hit_str = "Yes" if hit else "No"
    question_short = test["question"][:53]
    print(f"{question_short:<55} {test['expected_source']:<30} {rank_str:>5} {hit_str:>5}")

print()
print(f"Hit Rate:              {hits}/{len(test_queries)} ({hits/len(test_queries)*100:.0f}%)")
print(f"Mean Reciprocal Rank:  {sum(reciprocal_ranks)/len(reciprocal_ranks):.3f}")
print()
print("Hit Rate = fraction of queries where the right document appeared in top-k")
print("MRR = average of 1/rank (higher = the right document ranks higher)")
print("MRR of 1.0 = perfect (right document is always rank 1)")

**What this tells you:**

- **Hit Rate** measures whether the right document shows up at all in the top-k. If it is not there, no amount of prompt engineering can produce a correct answer.
- **MRR** measures how high the right document ranks. A right document at position 5 of 5 is less useful than one at position 1 (the "lost in the middle" effect).

These are simple metrics, but they give you a number to compare against. When we add metadata filtering in Session 3.2, we will run the same test queries and compare.

Note: these metrics only check retrieval. They do not check whether Claude's answer is correct. That is a harder problem -- one we address in Lab 2.

---

## 9. Bridge to Filtered RAG

Naive RAG treats all documents equally. Every query gets the same retrieval logic: embed the question, find the closest chunks, done.

But think about the failures we saw:

| Failure | Root Cause | What Would Fix It |
|---------|-----------|-------------------|
| Wrong version (2023 vs 2025 policy) | No temporal filtering | Filter by year or effective date |
| PTO carryover confusion | Conflicting documents treated equally | Filter by document type (memo vs policy) or recency |
| Over-retrieval on vague queries | No category scoping | Filter by doc_type before searching |
| Missing context across documents | Single-pass retrieval | Targeted retrieval from specific document types |

**What if we could tell the retriever what to look for?**

Instead of:
> "Find the 5 most similar chunks."

We could say:
> "Find the 5 most similar chunks **from HR policy documents written after 2024**."

That is metadata-aware retrieval. It uses the metadata you attached to chunks in Session 2.2 -- source, doc_type, date -- to **filter before searching**.

**Session 3.2 builds that pipeline. Lab 2 proves it works using data.**

The failures you documented today are the starting point for Lab 2. You already know where naive RAG breaks. Next session, you build the fix.

---

## 10. Session Recap

### What we explored today

1. **First RAG query** -- ran `naive_rag()` end-to-end and inspected the full `RAGResponse` with answer, sources, and token counts
2. **Under the hood** -- traced the pipeline stage by stage: `naive_retrieve()` for retrieval, `build_prompt()` for augmentation, and Claude for generation
3. **Answer vs Evidence panel** -- used `compare_answer_to_evidence()` to verify whether Claude's claims are supported by the retrieved chunks
4. **The Confidence Mirage** -- saw the most dangerous failure: high scores, confident answer, wrong result (PTO carryover conflict)
5. **Failure Gallery** -- systematically broke the pipeline across multiple failure categories
6. **Retrieval scoring** -- measured Hit Rate and MRR to quantify retrieval quality

### Key takeaways

- **Retrieval quality determines answer quality.** Wrong chunks in = wrong answer out.
- **Prompt design determines grounding.** Without explicit "say I don't know" instructions, Claude fills gaps from training data.
- **The evidence chain matters.** Question, chunks, scores, answer -- you need all of it to diagnose problems.
- **The dangerous failures look correct.** Low scores are easy to catch. High scores with wrong answers are the real threat.
- **Naive RAG is your floor.** It works for straightforward factual questions. It breaks on temporal, cross-document, and ambiguous queries.

### Questions to leave with

Think about these before Session 3.2:

- **On Filtering:** Every query gets the same retrieval logic -- what if the question type matters? Could we filter before we search?
- **On Comparison:** How would you prove that metadata filtering actually improves results? What does "better" even mean?
- **On Measurement:** If you had 100 questions, how would you score your RAG pipeline without reading every answer yourself?

### What is next

| Session | What We Build | Status |
|---------|--------------|--------|
| **1.1** | API connection + generation | Done |
| **1.2** | Structured extraction with Pydantic | Done |
| **2.1** | Embeddings + similarity measurement | Done |
| **2.2** | Chunking + vector store ingestion | Done |
| **3.1** | Naive RAG -- retrieve + generate | **Done (today)** |
| **3.2** | Metadata-filtered RAG + evaluation | Next session |
| **4.1** | Observability and debugging | Coming |
| **4.2** | Module test | Coming |

### Homework: Naive RAG Stress Test (not graded -- preparation for Lab 2)

Before next session:

1. Write **10 test questions** (3 should-work, 3 should-struggle, 2 ambiguous, 2 unanswerable)
2. Run all 10 through `naive_rag()` and record results
3. Rate each: **Good** (correct and grounded), **Partial** (partially correct), or **Bad** (wrong/hallucinated)
4. Experiment with `top_k=1, 3, 5, 10` on 2 questions

Save as `experiments/session_3_1_rag_stress_test.md`

**Lab 1 reminder:** Due at end of Session 3.2. If you have not started, start tonight.